In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Hello World: Primul tău Circuit Cuantic

Construiește un [stat Bell](https://en.wikipedia.org/wiki/Bell_state) (doi qubiți entanglați) și rulează-l în trei moduri:

1. **Simulare ideală** — rezultate perfecte, fără cont necesar
2. **Simulare cu zgomot** — simulează un dispozitiv real, fără cont necesar
3. **Hardware cuantic real** — necesită un [cont IBM Quantum](https://janlahmann.github.io/Qiskit-documentation/#setting-up-your-ibm-quantum-account)

## Construiește circuitul

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

qc.draw(output="mpl")

## Opțiunea 1: Simulare ideală (fără cont necesar)
Folosește `StatevectorSampler` — un simulator local cu rezultate perfecte, fără zgomot.

In [ ]:
from qiskit.primitives import StatevectorSampler

result = StatevectorSampler().run([qc], shots=1024).result()
counts = result[0].data.meas.get_counts()
counts

In [ ]:
from qiskit.visualization import plot_histogram
plot_histogram(counts)

## Opțiunea 2: Simulare cu zgomot (fără cont necesar)
Folosește `FakeManilaV2` — un simulator local care imită un dispozitiv cuantic IBM real, inclusiv caracteristicile sale de zgomot. Circuitul trebuie mai întâi transpilat (adaptat) la setul de porți și conectivitatea qubiților dispozitivului.

In [ ]:
from qiskit_ibm_runtime import SamplerV2
from qiskit_ibm_runtime.fake_provider import FakeManilaV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

backend = FakeManilaV2()
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_qc = pm.run(qc)

result = SamplerV2(mode=backend).run([isa_qc], shots=1024).result()
counts = result[0].data.meas.get_counts()
counts

In [ ]:
plot_histogram(counts)

## Opțiunea 3: Hardware cuantic real
Necesită un cont IBM Quantum. Consultă [Configurarea contului tău IBM Quantum](https://janlahmann.github.io/Qiskit-documentation/#setting-up-your-ibm-quantum-account) pentru detalii.

Dacă nu ți-ai salvat încă credențialele în această sesiune Binder, rulează mai întâi aceasta:

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
    token="<your-api-key>",
    instance="<your-crn>",
    overwrite=True
)

**Notă:** Job-urile pe hardware real pot dura ceva timp în funcție de timpii de așteptare în coadă. Dacă celula încă rulează, poți verifica starea job-ului și vedea rezultatele la [quantum.cloud.ibm.com/workloads](https://quantum.cloud.ibm.com/workloads?user=me).

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)
print(f"Running on {backend.name}")

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_qc = pm.run(qc)

result = SamplerV2(mode=backend).run([isa_qc], shots=1024).result()
counts = result[0].data.meas.get_counts()
counts

In [ ]:
plot_histogram(counts)

## Ce urmează?
- **[Tutoriale](https://mybinder.org/v2/gh/JanLahmann/Qiskit-documentation/main?filepath=docs/tutorials)** — ghiduri pas cu pas despre algoritmi, atenuarea erorilor, transpilare și altele
- **[Cursuri](https://mybinder.org/v2/gh/JanLahmann/Qiskit-documentation/main?filepath=learning/courses)** — trasee de învățare structurate, de la bazele cuantice până la computație la scară utilitară
- **[Modul de testare locală](https://janlahmann.github.io/Qiskit-documentation/#no-token-use-local-testing-mode)** — rulează majoritatea notebook-urilor fără un cont IBM Quantum